In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler

from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

In [2]:
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"

os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
cleaned_orders = pd.read_csv(f"{PROCESSED_DIR}/cleaned_orders.csv", parse_dates=["order_date", "shipping_date", "week_start"])
demand_ts = pd.read_csv(f"{PROCESSED_DIR}/demand_timeseries.csv", parse_dates=["week_start"])

print(cleaned_orders.shape)
print(demand_ts.shape)

display(cleaned_orders.head())
display(demand_ts.head())

(171962, 45)
(16781, 13)


,order_date,shipping_date,week_start,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Category Id,Category Name,Customer City,...,Shipping Mode,year,month,day,dayofweek,week,quarter,product_idx,category_idx,region_idx
0,2016-02-24 13:57:00,2016-02-29 13:57:00,2016-02-22,PAYMENT,2,-30.750000,115.180000,17,Cleats,Bayamon,...,Second Class,2016,2,24,2,8,1,0,0,0
1,2016-10-25 14:39:00,2016-10-27 14:39:00,2016-10-24,PAYMENT,2,-122.730003,79.180000,29,Shop By Sport,Caguas,...,Second Class,2016,10,25,1,43,4,1,1,1
2,2016-03-30 04:37:00,2016-04-05 04:37:00,2016-03-28,PAYMENT,2,33.599998,96.000000,24,Women's Apparel,Caguas,...,Second Class,2016,3,30,2,13,1,2,2,2
3,2016-10-30 01:31:00,2016-11-01 01:31:00,2016-10-24,PAYMENT,2,24.690001,75.980003,29,Shop By Sport,Caguas,...,Second Class,2016,10,30,6,43,4,1,1,1
4,2016-11-28 01:18:00,2016-12-01 01:18:00,2016-11-28,PAYMENT,2,9.100000,91.000000,24,Women's Apparel,Caguas,...,Second Class,2016,11,28,0,48,4,2,2,3


,Product Card Id,Product Name,Category Id,Category Name,Order Region,week_start,demand,avg_price,avg_discount,avg_sales,avg_profit,shipping_mode_nunique,region_order_count
0,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,Caribbean,2017-05-01,1,124.989998,6.25,124.989998,-86.089996,1,1
1,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,Caribbean,2017-05-08,1,124.989998,5.00,124.989998,34.799999,1,1
2,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,Caribbean,2017-05-15,1,124.989998,3.75,124.989998,9.090000,1,1
3,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,Caribbean,2017-05-22,2,124.989998,21.25,124.989998,39.275000,1,2
4,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,Caribbean,2017-06-05,1,124.989998,12.50,124.989998,34.869999,1,1


In [5]:
product_ids = cleaned_orders["Product Card Id"].drop_duplicates().tolist()
category_ids = cleaned_orders["Category Id"].drop_duplicates().tolist()
region_ids = cleaned_orders["Order Region"].drop_duplicates().tolist()

product_map = {v: i for i, v in enumerate(product_ids)}
category_map = {v: i for i, v in enumerate(category_ids)}
region_map = {v: i for i, v in enumerate(region_ids)}

print("Products:", len(product_map))
print("Categories:", len(category_map))
print("Regions:", len(region_map))

Products: 100
Categories: 33
Regions: 23


In [6]:
product_feat_df = (
    cleaned_orders.groupby("Product Card Id", as_index=False)
    .agg(
        product_price_mean=("Product Price", "mean"),
        sales_mean=("Sales", "mean"),
        qty_mean=("Order Item Quantity", "mean"),
        profit_mean=("Order Profit Per Order", "mean"),
        discount_mean=("Order Item Discount", "mean"),
        unique_regions=("Order Region", "nunique"),
        unique_categories=("Category Id", "nunique"),
    )
)

product_feat_df["product_idx"] = product_feat_df["Product Card Id"].map(product_map)
product_feat_df = product_feat_df.sort_values("product_idx").reset_index(drop=True)

product_feature_cols = [
    "product_price_mean",
    "sales_mean",
    "qty_mean",
    "profit_mean",
    "discount_mean",
    "unique_regions",
    "unique_categories",
]

product_scaler = StandardScaler()
product_x = product_scaler.fit_transform(product_feat_df[product_feature_cols].fillna(0))
product_x = torch.tensor(product_x, dtype=torch.float)

print(product_x.shape)
display(product_feat_df.head())

torch.Size([100, 7])


,Product Card Id,product_price_mean,sales_mean,qty_mean,profit_mean,discount_mean,unique_regions,unique_categories,product_idx
0,365,59.990002,180.322916,3.005883,20.135436,18.277405,23,1,0
1,627,39.990002,119.540245,2.989253,11.879072,12.088334,23,1,1
2,502,50.000000,149.652513,2.993050,16.672128,15.171911,23,1,2
3,278,44.990002,130.631681,2.903571,13.376214,13.684750,21,1,3
4,249,54.970001,162.228540,2.951220,15.574112,16.119965,23,1,4


In [7]:
category_feat_df = (
    cleaned_orders.groupby("Category Id", as_index=False)
    .agg(
        category_price_mean=("Product Price", "mean"),
        category_sales_mean=("Sales", "mean"),
        category_qty_mean=("Order Item Quantity", "mean"),
        category_profit_mean=("Order Profit Per Order", "mean"),
        category_unique_products=("Product Card Id", "nunique"),
    )
)

category_feat_df["category_idx"] = category_feat_df["Category Id"].map(category_map)
category_feat_df = category_feat_df.sort_values("category_idx").reset_index(drop=True)

category_feature_cols = [
    "category_price_mean",
    "category_sales_mean",
    "category_qty_mean",
    "category_profit_mean",
    "category_unique_products",
]

category_scaler = StandardScaler()
category_x = category_scaler.fit_transform(category_feat_df[category_feature_cols].fillna(0))
category_x = torch.tensor(category_x, dtype=torch.float)

print(category_x.shape)
display(category_feat_df.head())

torch.Size([33, 5])


,Category Id,category_price_mean,category_sales_mean,category_qty_mean,category_profit_mean,category_unique_products,category_idx
0,17,60.342453,180.498653,3.002937,20.138852,2,0
1,29,40.631737,119.227544,2.979590,11.803977,3,1
2,24,50.000000,149.652513,2.993050,16.672128,1,2
3,13,34.328953,102.356034,2.994676,10.882786,4,3
4,12,67.056809,201.431232,2.990544,20.428771,3,4


In [8]:
#product to category
prod_cat = (
    cleaned_orders[["Product Card Id", "Category Id"]]
    .drop_duplicates()
    .copy()
)

prod_cat["product_idx"] = prod_cat["Product Card Id"].map(product_map)
prod_cat["category_idx"] = prod_cat["Category Id"].map(category_map)

product_category_edge_index = torch.tensor(
    prod_cat[["product_idx", "category_idx"]].values.T,
    dtype=torch.long
)

print(product_category_edge_index.shape)

torch.Size([2, 100])


In [9]:
#product to region 
prod_region = (
    cleaned_orders[["Product Card Id", "Order Region"]]
    .drop_duplicates()
    .copy()
)

prod_region["product_idx"] = prod_region["Product Card Id"].map(product_map)
prod_region["region_idx"] = prod_region["Order Region"].map(region_map)

product_region_edge_index = torch.tensor(
    prod_region[["product_idx", "region_idx"]].values.T,
    dtype=torch.long
)

print(product_region_edge_index.shape)

torch.Size([2, 1444])


In [11]:
#building hetero data 
data = HeteroData()

data["product"].x = product_x
data["category"].x = category_x
# Define region node features (one-hot identity) before attaching to HeteroData
region_x = torch.eye(len(region_map), dtype=torch.float)
data["region"].x = region_x

data["product", "belongs_to", "category"].edge_index = product_category_edge_index
data["category", "has_product", "product"].edge_index = product_category_edge_index.flip(0)

data["product", "sold_in", "region"].edge_index = product_region_edge_index
data["region", "contains_product", "product"].edge_index = product_region_edge_index.flip(0)

print(data)

HeteroData(
  product={ x=[100, 7] },
  category={ x=[33, 5] },
  region={ x=[23, 23] },
  (product, belongs_to, category)={ edge_index=[2, 100] },
  (category, has_product, product)={ edge_index=[2, 100] },
  (product, sold_in, region)={ edge_index=[2, 1444] },
  (region, contains_product, product)={ edge_index=[2, 1444] }
)


In [13]:
#saving the grpah and embeddings
#this will be helpful for future use without needing to reprocess everything
torch.save(data, f"{PROCESSED_DIR}/graph_data.pt")

region_feature_cols = [f"region_{region}" for region in region_ids]

joblib.dump(
    {
        "product_map": product_map,
        "category_map": category_map,
        "region_map": region_map,
        "product_feature_cols": product_feature_cols,
        "category_feature_cols": category_feature_cols,
        "region_feature_cols": region_feature_cols,
    },
    f"{PROCESSED_DIR}/node_mappings.joblib"
)

print("Saved graph and mappings.")

Saved graph and mappings.


In [14]:
class HeteroGraphSAGE(nn.Module):
    def __init__(self, metadata, hidden_dim=64, out_dim=32):
        super().__init__()
        self.lin_dict = nn.ModuleDict()
        for node_type in metadata[0]:
            self.lin_dict[node_type] = nn.LazyLinear(hidden_dim)

        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_dim)
            for edge_type in metadata[1]
        }, aggr="mean")

        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), out_dim)
            for edge_type in metadata[1]
        }, aggr="mean")

    def forward(self, x_dict, edge_index_dict):
        x_dict = {
            node_type: self.lin_dict[node_type](x)
            for node_type, x in x_dict.items()
        }

        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}

        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict

In [15]:
#helper for link prediction 
def decode_product_region(z_product, z_region, edge_index):
    src = z_product[edge_index[0]]
    dst = z_region[edge_index[1]]
    return (src * dst).sum(dim=-1)

def negative_sampling_product_region(num_products, num_regions, num_samples, device):
    src = torch.randint(0, num_products, (num_samples,), device=device)
    dst = torch.randint(0, num_regions, (num_samples,), device=device)
    return torch.stack([src, dst], dim=0)

In [16]:
#training loop for link prediction through graphsage 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph = data.to(device)

model = HeteroGraphSAGE(graph.metadata(), hidden_dim=64, out_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

pos_edge_index = graph["product", "sold_in", "region"].edge_index
num_products = graph["product"].x.size(0)
num_regions = graph["region"].x.size(0)

print("Using device:", device)
print("Positive edges:", pos_edge_index.size(1))

Using device: cuda
Positive edges: 1444


In [20]:
#training loop
epochs = 10000

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    z_dict = model(graph.x_dict, graph.edge_index_dict)

    pos_logits = decode_product_region(
        z_dict["product"], z_dict["region"], pos_edge_index
    )

    neg_edge_index = negative_sampling_product_region(
        num_products=num_products,
        num_regions=num_regions,
        num_samples=pos_edge_index.size(1),
        device=device
    )

    neg_logits = decode_product_region(
        z_dict["product"], z_dict["region"], neg_edge_index
    )

    pos_labels = torch.ones_like(pos_logits)
    neg_labels = torch.zeros_like(neg_logits)

    logits = torch.cat([pos_logits, neg_logits], dim=0)
    labels = torch.cat([pos_labels, neg_labels], dim=0)

    loss = F.binary_cross_entropy_with_logits(logits, labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            acc = (preds == labels).float().mean().item()

        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

Epoch 001 | Loss: 0.5390 | Acc: 0.6918
Epoch 010 | Loss: 0.5485 | Acc: 0.6807
Epoch 020 | Loss: 0.5576 | Acc: 0.6717
Epoch 030 | Loss: 0.5361 | Acc: 0.6943
Epoch 040 | Loss: 0.5480 | Acc: 0.6821
Epoch 050 | Loss: 0.5528 | Acc: 0.6769
Epoch 060 | Loss: 0.5442 | Acc: 0.6842
Epoch 070 | Loss: 0.5403 | Acc: 0.6915
Epoch 080 | Loss: 0.5482 | Acc: 0.6825
Epoch 090 | Loss: 0.5556 | Acc: 0.6724
Epoch 100 | Loss: 0.5446 | Acc: 0.6839
Epoch 110 | Loss: 0.5533 | Acc: 0.6749
Epoch 120 | Loss: 0.5493 | Acc: 0.6794
Epoch 130 | Loss: 0.5519 | Acc: 0.6780
Epoch 140 | Loss: 0.5520 | Acc: 0.6769
Epoch 150 | Loss: 0.5517 | Acc: 0.6766
Epoch 160 | Loss: 0.5380 | Acc: 0.6946
Epoch 170 | Loss: 0.5498 | Acc: 0.6807
Epoch 180 | Loss: 0.5524 | Acc: 0.6783
Epoch 190 | Loss: 0.5598 | Acc: 0.6690
Epoch 200 | Loss: 0.5383 | Acc: 0.6894
Epoch 210 | Loss: 0.5514 | Acc: 0.6762
Epoch 220 | Loss: 0.5424 | Acc: 0.6887
Epoch 230 | Loss: 0.5476 | Acc: 0.6842
Epoch 240 | Loss: 0.5417 | Acc: 0.6884
Epoch 250 | Loss: 0.5435 

In [21]:
def negative_sampling_product_region(pos_edge_index, num_products, num_regions, num_samples, device):
    pos_set = set((int(s), int(d)) for s, d in zip(pos_edge_index[0].cpu().tolist(),
                                                   pos_edge_index[1].cpu().tolist()))
    
    neg_src = []
    neg_dst = []

    while len(neg_src) < num_samples:
        s = torch.randint(0, num_products, (1,)).item()
        d = torch.randint(0, num_regions, (1,)).item()

        if (s, d) not in pos_set:
            neg_src.append(s)
            neg_dst.append(d)

    return torch.tensor([neg_src, neg_dst], dtype=torch.long, device=device)

In [22]:
neg_edge_index = negative_sampling_product_region(
    pos_edge_index=pos_edge_index,
    num_products=num_products,
    num_regions=num_regions,
    num_samples=pos_edge_index.size(1),
    device=device
)

In [23]:
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# -----------------------------
# 1) SAFE TRAIN/VAL EDGE SPLIT
# -----------------------------
edges = pos_edge_index.cpu().numpy().T
train_edges, val_edges = train_test_split(edges, test_size=0.2, random_state=42)

train_pos_edge_index = torch.tensor(train_edges.T, dtype=torch.long, device=device)
val_pos_edge_index = torch.tensor(val_edges.T, dtype=torch.long, device=device)

# -----------------------------
# 2) SAFER NEGATIVE SAMPLING
#    avoids collisions with real edges
# -----------------------------
def negative_sampling_product_region(pos_edge_index, num_products, num_regions, num_samples, device):
    pos_set = set(
        (int(s), int(d))
        for s, d in zip(
            pos_edge_index[0].detach().cpu().tolist(),
            pos_edge_index[1].detach().cpu().tolist()
        )
    )

    neg_src = []
    neg_dst = []

    while len(neg_src) < num_samples:
        s = torch.randint(0, num_products, (1,)).item()
        d = torch.randint(0, num_regions, (1,)).item()

        if (s, d) not in pos_set:
            neg_src.append(s)
            neg_dst.append(d)

    return torch.tensor([neg_src, neg_dst], dtype=torch.long, device=device)

# -----------------------------
# 3) NORMALIZED DECODER
#    prevents exploding dot products
# -----------------------------
def decode_product_region(z_product, z_region, edge_index):
    src = z_product[edge_index[0]]
    dst = z_region[edge_index[1]]

    src = F.normalize(src, p=2, dim=-1)
    dst = F.normalize(dst, p=2, dim=-1)

    return (src * dst).sum(dim=-1)

# -----------------------------
# 4) LOWER LR + GRAD CLIPPING
# -----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

# -----------------------------
# UPDATED TRAINING LOOP
# -----------------------------
epochs = 200

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    z_dict = model(graph.x_dict, graph.edge_index_dict)

    # positive training edges
    pos_logits = decode_product_region(
        z_dict["product"],
        z_dict["region"],
        train_pos_edge_index
    )

    # negative training edges
    neg_edge_index = negative_sampling_product_region(
        pos_edge_index=train_pos_edge_index,
        num_products=num_products,
        num_regions=num_regions,
        num_samples=train_pos_edge_index.size(1),
        device=device
    )

    neg_logits = decode_product_region(
        z_dict["product"],
        z_dict["region"],
        neg_edge_index
    )

    pos_labels = torch.ones_like(pos_logits)
    neg_labels = torch.zeros_like(neg_logits)

    train_logits = torch.cat([pos_logits, neg_logits], dim=0)
    train_labels = torch.cat([pos_labels, neg_labels], dim=0)

    train_loss = F.binary_cross_entropy_with_logits(train_logits, train_labels)

    train_loss.backward()

    # gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            z_dict = model(graph.x_dict, graph.edge_index_dict)

            # validation positives
            val_pos_logits = decode_product_region(
                z_dict["product"],
                z_dict["region"],
                val_pos_edge_index
            )

            # validation negatives
            val_neg_edge_index = negative_sampling_product_region(
                pos_edge_index=val_pos_edge_index,
                num_products=num_products,
                num_regions=num_regions,
                num_samples=val_pos_edge_index.size(1),
                device=device
            )

            val_neg_logits = decode_product_region(
                z_dict["product"],
                z_dict["region"],
                val_neg_edge_index
            )

            val_logits = torch.cat([val_pos_logits, val_neg_logits], dim=0)
            val_labels = torch.cat([
                torch.ones_like(val_pos_logits),
                torch.zeros_like(val_neg_logits)
            ], dim=0)

            val_loss = F.binary_cross_entropy_with_logits(val_logits, val_labels)

            train_probs = torch.sigmoid(train_logits)
            train_preds = (train_probs > 0.5).float()
            train_acc = (train_preds == train_labels).float().mean().item()

            val_probs = torch.sigmoid(val_logits)
            val_preds = (val_probs > 0.5).float()
            val_acc = (val_preds == val_labels).float().mean().item()

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss.item():.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss.item():.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

Epoch 001 | Train Loss: 0.5355 | Train Acc: 0.8606 | Val Loss: 0.6026 | Val Acc: 0.7111
Epoch 010 | Train Loss: 0.5157 | Train Acc: 0.8524 | Val Loss: 0.5975 | Val Acc: 0.7042
Epoch 020 | Train Loss: 0.4882 | Train Acc: 0.8459 | Val Loss: 0.6013 | Val Acc: 0.7093
Epoch 030 | Train Loss: 0.5045 | Train Acc: 0.8152 | Val Loss: 0.6040 | Val Acc: 0.6886
Epoch 040 | Train Loss: 0.4886 | Train Acc: 0.8273 | Val Loss: 0.6111 | Val Acc: 0.6817
Epoch 050 | Train Loss: 0.4822 | Train Acc: 0.8472 | Val Loss: 0.5992 | Val Acc: 0.6990
Epoch 060 | Train Loss: 0.4863 | Train Acc: 0.8532 | Val Loss: 0.5856 | Val Acc: 0.7145
Epoch 070 | Train Loss: 0.4894 | Train Acc: 0.8511 | Val Loss: 0.5989 | Val Acc: 0.6955
Epoch 080 | Train Loss: 0.4809 | Train Acc: 0.8628 | Val Loss: 0.5992 | Val Acc: 0.7042
Epoch 090 | Train Loss: 0.4812 | Train Acc: 0.8610 | Val Loss: 0.5936 | Val Acc: 0.7180
Epoch 100 | Train Loss: 0.4882 | Train Acc: 0.8519 | Val Loss: 0.5919 | Val Acc: 0.7163
Epoch 110 | Train Loss: 0.4815 |

In [24]:
model.eval()

with torch.no_grad():
    z_dict = model(graph.x_dict, graph.edge_index_dict)

product_embeddings = z_dict["product"].cpu().numpy()
region_embeddings = z_dict["region"].cpu().numpy()
category_embeddings = z_dict["category"].cpu().numpy()

print(product_embeddings.shape)
print(region_embeddings.shape)
print(category_embeddings.shape)

(100, 32)
(23, 32)
(33, 32)


In [25]:
product_emb_df = pd.DataFrame(product_embeddings)
product_emb_df["product_idx"] = range(len(product_emb_df))

region_emb_df = pd.DataFrame(region_embeddings)
region_emb_df["region_idx"] = range(len(region_emb_df))

product_emb_df.to_csv("../data/processed/product_embeddings.csv", index=False)
region_emb_df.to_csv("../data/processed/region_embeddings.csv", index=False)